# ChronoState-KD Lite — Phase 3: Adapter store + weight jump

**Goal:** keep dormant adapters out of VRAM in pinned host memory and swap the active one by overwriting the live LoRA tensors in place (`copy_()`), then measure it.

**Budget:** ~15 min on a T4.

**Gate to pass before Phase 4:**
1. Correctness: after `jump("reasoning")` the greedy output is **identical** to a cold `PeftModel.from_pretrained(adapter-reasoning)` on 3 prompts (fp32 store)
2. Jump latency **p95 < 45 ms** over 200 alternating jumps; no CUDA memory allocated inside the loop
3. Baseline for comparison: time of the naive path (`load_adapter` from local disk + `set_adapter`)

Runtime → **T4 GPU**. Edit `GITHUB_URL`.

In [ ]:
GITHUB_URL = "https://github.com/<YOUR_GITHUB_USER>/chronostate-kd.git"   # <-- edit
REPO_DIR   = "/content/chronostate-kd"
DRIVE_ROOT = "/content/drive/MyDrive/chronostate"
LOCAL_ADAPTERS = "/content/adapters"       # Drive is slow; copy adapters to local disk once so the baseline is fair
ADAPTERS = {"code": "adapter-code", "reasoning": "adapter-reasoning"}


## 1. Setup

In [ ]:
import os, json, time, gc, shutil, torch
assert torch.cuda.is_available(), "Runtime -> Change runtime type -> T4 GPU"
print(torch.cuda.get_device_name(0))
if not os.path.exists(REPO_DIR):
    !git clone $GITHUB_URL $REPO_DIR
%cd $REPO_DIR
!git pull --ff-only
!pip install -q -r requirements.txt && pip install -q -e .
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(LOCAL_ADAPTERS, exist_ok=True)
for run in ADAPTERS.values():
    if not os.path.exists(f"{LOCAL_ADAPTERS}/{run}"):
        shutil.copytree(f"{DRIVE_ROOT}/adapters/{run}", f"{LOCAL_ADAPTERS}/{run}")
!du -sh $LOCAL_ADAPTERS/*
from chronostate.config import load_config
from chronostate.monitor import Monitor
from chronostate.store import AdapterStore
from chronostate.jump import WeightJumper, greedy_ids, equivalence_report
from chronostate.data import SYSTEM_PROMPTS
cfg = load_config("configs/lite.yaml")
mon = Monitor(f"{DRIVE_ROOT}/results/events.jsonl", run="phase3-jump")


## 2. Load the student once, with one adapter attached as the container
The attached adapter's tensors are the GPU slots every later jump overwrites. Which adapter is attached first does not matter.

In [ ]:
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
tok = AutoTokenizer.from_pretrained(cfg.models.student)
base = AutoModelForCausalLM.from_pretrained(cfg.models.student, quantization_config=bnb, attn_implementation="sdpa", dtype=torch.float16, device_map={"": 0})
model = PeftModel.from_pretrained(base, f"{LOCAL_ADAPTERS}/{ADAPTERS['code']}", adapter_name="default")
model.eval()
n_lora = sum(p.numel() for n, p in model.named_parameters() if ".lora_" in n)
print(f"LoRA params in live slots: {n_lora:,}  VRAM {torch.cuda.memory_allocated()/1e9:.2f} GB")


## 3. Build the stores (fp32 = exact, fp16 = half the bytes)

In [ ]:
store32 = AdapterStore(dtype=None).add_all({k: f"{LOCAL_ADAPTERS}/{v}" for k, v in ADAPTERS.items()})
store16 = AdapterStore(dtype=torch.float16).add_all({k: f"{LOCAL_ADAPTERS}/{v}" for k, v in ADAPTERS.items()})
print(store32); print(store16)
for n, m in store32.meta.items(): mon.log("store_load", store="fp32", adapter=n, **m)
for n, m in store16.meta.items(): mon.log("store_load", store="fp16", adapter=n, **m)
jumper = WeightJumper(model, store32)
print("live tensors:", len(jumper.live), " bytes per jump:", f"{jumper.nbytes/1e6:.1f} MB")


## 4. Correctness: jumped weights == cold-loaded adapter
Same prompt, greedy decoding, 48 tokens. The reference is a *separate* `PeftModel.from_pretrained` of the same adapter.

In [ ]:
probes = [("reasoning", "A train travels 60 km in 1.5 hours. What is its average speed in km/h?"),
          ("code", "Write a Python function is_palindrome(s) that ignores case and spaces."),
          ("reasoning", "Maya has 3 boxes with 12 apples each and gives away 7. How many apples are left?")]
reports = []
for domain, prompt in probes:
    jumper.jump(domain)
    jumped = greedy_ids(model, tok, prompt, SYSTEM_PROMPTS[domain])
    # reference: fresh PEFT load of the same adapter on a fresh base (fully independent path)
    ref_base = AutoModelForCausalLM.from_pretrained(cfg.models.student, quantization_config=bnb, attn_implementation="sdpa", dtype=torch.float16, device_map={"": 0})
    ref = PeftModel.from_pretrained(ref_base, f"{LOCAL_ADAPTERS}/{ADAPTERS[domain]}").eval()
    reference = greedy_ids(ref, tok, prompt, SYSTEM_PROMPTS[domain])
    del ref, ref_base; gc.collect(); torch.cuda.empty_cache()
    rep = equivalence_report(jumped, reference); rep.update(domain=domain, prompt=prompt[:40])
    reports.append(rep); mon.log("jump_equivalence", **rep)
    print(rep); print("   ", repr(tok.decode(jumped)[:120]))
assert all(r["identical"] for r in reports), "jumped output differs from cold load - STOP"
print("all identical")


## 5. Benchmark: 200 alternating jumps, fp32 store then fp16 store
`cuda_mem_delta` must be 0 on every jump: nothing is allocated inside the loop.

In [ ]:
torch.cuda.synchronize()
bench = {}
bench["fp32"] = jumper.benchmark(["code", "reasoning"], trials=cfg.eval.jump_trials, warmup=5, monitor=mon, tag="fp32")
jumper16 = WeightJumper(model, store16)
bench["fp16"] = jumper16.benchmark(["code", "reasoning"], trials=cfg.eval.jump_trials, warmup=5, monitor=mon, tag="fp16")
for k, b in bench.items():
    print(f"{k}: p50 {b['p50_ms']:.2f} ms  p95 {b['p95_ms']:.2f} ms  p99 {b['p99_ms']:.2f} ms  max {b['max_ms']:.2f} ms | "
          f"fetch {b['fetch_p50_ms']:.3f} copy {b['copy_p50_ms']:.2f} sync {b['sync_p50_ms']:.2f} | {b['mb_per_jump']:.1f} MB/jump, {b['gbps']:.1f} GB/s | mem delta {b['max_cuda_mem_delta']}")
assert bench["fp32"]["max_cuda_mem_delta"] == 0 and bench["fp16"]["max_cuda_mem_delta"] == 0
target = cfg.eval.jump_p95_target_ms
print("TARGET p95 <", target, "ms ->", "PASS" if min(b["p95_ms"] for b in bench.values()) < target else "FAIL")


## 6. Baseline: the naive swap path
What multi-adapter code usually does: `load_adapter()` from disk each time, then `set_adapter()`. Measured from **local** disk, so the comparison is about the code path, not Drive latency.

In [ ]:
times = []
for i in range(10):
    name = f"tmp{i}"; src = ADAPTERS["reasoning" if i % 2 else "code"]
    torch.cuda.synchronize(); t0 = time.perf_counter()
    model.load_adapter(f"{LOCAL_ADAPTERS}/{src}", adapter_name=name)
    model.set_adapter(name)
    torch.cuda.synchronize(); times.append((time.perf_counter() - t0) * 1e3)
    model.delete_adapter(name)
model.set_adapter("default")
import statistics
baseline = {"trials": len(times), "p50_ms": statistics.median(times), "min_ms": min(times), "max_ms": max(times)}
print("naive load_adapter+set_adapter:", baseline)
mon.log("naive_swap_benchmark", **baseline)
print(f"speedup p50: {baseline['p50_ms']/bench['fp32']['p50_ms']:.0f}x (fp32 store), {baseline['p50_ms']/bench['fp16']['p50_ms']:.0f}x (fp16 store)")


## 7. Plot + save

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist([e["t_total_ms"] for e in jumper.events], bins=40, alpha=0.7, label=f"fp32 store ({bench['fp32']['mb_per_jump']:.0f} MB)")
ax.hist([e["t_total_ms"] for e in jumper16.events], bins=40, alpha=0.7, label=f"fp16 store ({bench['fp16']['mb_per_jump']:.0f} MB)")
ax.axvline(target, color="red", ls="--", label=f"target p95 < {target} ms")
ax.set_xlabel("jump latency (ms)"); ax.set_ylabel("count"); ax.set_title("In-place adapter swap, 200 jumps each, Tesla T4"); ax.legend()
os.makedirs("results", exist_ok=True)
plt.tight_layout(); plt.savefig("results/phase3_jump_latency.png", dpi=130); plt.show()
out = {"gpu": torch.cuda.get_device_name(0), "lora_params": n_lora, "equivalence": reports, "benchmark": bench,
       "naive_baseline_ms": baseline, "store_meta": {"fp32": store32.meta, "fp16": store16.meta}}
json.dump(out, open("results/phase3_jump_benchmark.json", "w"), indent=2)
json.dump(out, open(f"{DRIVE_ROOT}/results/phase3_jump_benchmark.json", "w"), indent=2)
mon.close(); print(json.dumps({k: v for k, v in out.items() if k != "equivalence"}, indent=2))


## Gate checklist
- [ ] 3/3 equivalence reports `identical: True`
- [ ] p95 < 45 ms for at least the fp16 store (report both)
- [ ] `max_cuda_mem_delta == 0`
- [ ] `results/phase3_jump_benchmark.json` + `results/phase3_jump_latency.png` downloaded and committed

Report back: the two benchmark lines, the baseline line, the speed-up line.